# HitPredict: Google Colab demo
**Team 21 · UE24CS352A · E Section**  
N S RAJ NANJAPPA (PES1UG24CS287) and NAGARAJ HEGDE (PES1UG24CS288)

Run **Runtime > Run all**. The last cells display a song selector, editable audio features, a prediction button, and batch results. A standard CPU runtime is sufficient.

This notebook includes the cleaned dataset and original split assignments. It needs no GitHub token, Drive mount, or separate data upload. It fits the selected random forest inside the current runtime and keeps the original validation-selected cutoff of 0.29. All displayed runtime metrics are recomputed. Dependency versions can produce small differences from the submitted experiment.

The model classifies inherited historical Billboard labels. Its scores are not calibrated probabilities of future chart success.

## 1. Prepare the runtime

In [ ]:
import importlib.util
import subprocess
import sys

# Use the runtime's installed scientific libraries. Install only missing packages.
packages = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn',
            'matplotlib': 'matplotlib', 'ipywidgets': 'ipywidgets'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])

import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('Runtime ready. No GPU is required.')

## 2. Load the demo functions and embedded data
The functions below fit only the selected random forest configuration, validate input features, and provide notebook controls. The next folded cell contains a compressed data snapshot with a checksum. The dataset is the same 4,019 cleaned records used in the project, including the saved train/validation/test assignments.

In [ ]:
"""Standalone notebook demo logic, also embedded as readable Colab notebook code."""
import base64
import hashlib
import io
import json
import zlib

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

DEMO_FEATURES = ["Danceability", "Energy", "Loudness", "Speechiness", "Acousticness",
                 "Instrumentalness", "Liveness", "Valence", "Tempo"]

def decode_demo_assets(encoded, expected_hash):
    """Verify the embedded data snapshot before reading its source-defined splits."""
    packed = base64.b64decode(encoded)
    if hashlib.sha256(packed).hexdigest() != expected_hash:
        raise ValueError("The embedded data checksum does not match.")
    payload = json.loads(zlib.decompress(packed))
    tracks = pd.read_csv(io.StringIO(payload["tracks_csv"]))
    required = set(DEMO_FEATURES + ["row_id", "Artist", "Track", "Label", "group", "split"])
    if not required.issubset(tracks.columns):
        raise ValueError("The embedded dataset is incomplete.")
    if not tracks.row_id.is_unique or not tracks.Label.isin([0, 1]).all():
        raise ValueError("Invalid source identities or labels.")
    if set(tracks.split) != {"train", "validation", "test"}:
        raise ValueError("Unexpected split names.")
    for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
        if not set(tracks.loc[tracks.split == a, "group"]).isdisjoint(
                tracks.loc[tracks.split == b, "group"]):
            raise ValueError("An artist/audio group crosses evaluation splits.")
    return tracks, payload["reference"]

def fit_demo_model(tracks, reference):
    """Refit only the selected configuration on the original training partition."""
    train = tracks.loc[tracks.split == "train"]
    forest = RandomForestClassifier(n_estimators=250, max_depth=16,
        min_samples_leaf=5, random_state=21, n_jobs=1)
    model = Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler()), ("model", forest)])
    model.fit(train[DEMO_FEATURES], train.Label)
    return {"model": model, "threshold": float(reference["threshold"]),
            "training_medians": train[DEMO_FEATURES].median().to_dict()}

def score_demo(frame, bundle):
    """Validate explicit inputs and score them with the current notebook model."""
    missing = set(DEMO_FEATURES) - set(frame.columns)
    if missing:
        raise ValueError("Missing columns: " + ", ".join(sorted(missing)))
    x = frame[DEMO_FEATURES].apply(pd.to_numeric, errors="raise").astype(float)
    if x.empty or not np.isfinite(x.to_numpy()).all():
        raise ValueError("Provide at least one track with finite numeric audio features.")
    for feature in DEMO_FEATURES:
        low, high = (-60., 5.) if feature == "Loudness" else ((0., 300.) if feature == "Tempo" else (0., 1.))
        if not x[feature].between(low, high).all():
            raise ValueError(f"{feature} must be between {low:g} and {high:g}.")
    result = frame.copy()
    result["hit_score"] = bundle["model"].predict_proba(x)[:, 1]
    result["predicted_label"] = (result.hit_score >= bundle["threshold"]).astype(int)
    return result

def evaluate_demo(tracks, bundle):
    test = tracks.loc[tracks.split == "test"]
    scored = score_demo(test, bundle)
    y, pred, score = scored.Label, scored.predicted_label, scored.hit_score
    metrics = {"accuracy": float(accuracy_score(y, pred)),
               "precision": float(precision_score(y, pred, zero_division=0)),
               "recall": float(recall_score(y, pred, zero_division=0)),
               "f1": float(f1_score(y, pred, zero_division=0)),
               "roc_auc": float(roc_auc_score(y, score)),
               "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist()}
    return scored, metrics

def create_demo_controls(examples, bundle):
    """Build notebook-native controls without a web server or external tunnel."""
    import ipywidgets as widgets
    from IPython.display import display

    examples = examples.reset_index(drop=True)
    options = [(f"{row.Artist} - {row.Track}", i) for i, row in examples.iterrows()]
    options.append(("Custom track", -1))
    selector = widgets.Dropdown(options=options, value=0, description="Track:",
                                layout=widgets.Layout(width="95%"))
    source_label = widgets.HTML()
    fields = {}
    for feature in DEMO_FEATURES:
        low, high, step = (-60., 5., .1) if feature == "Loudness" else ((0., 300., 1.) if feature == "Tempo" else (0., 1., .001))
        unit = " (dB)" if feature == "Loudness" else " (BPM)" if feature == "Tempo" else ""
        fields[feature] = widgets.BoundedFloatText(value=float(examples.iloc[0][feature]),
            min=low, max=high, step=step, description=feature + unit,
            style={"description_width": "160px"}, layout=widgets.Layout(width="330px"))
    output = widgets.Output()
    state = {"last_result": None, "last_error": None}
    def load_example(change=None):
        selected = selector.value
        values = bundle["training_medians"] if selected == -1 else examples.iloc[selected]
        for feature in DEMO_FEATURES:
            fields[feature].value = float(values[feature])
        source_label.value = ("Custom inputs start at training-set medians." if selected == -1 else
            "Source label for the original example: <b>" + ("Hit" if int(values.Label) else "Non-hit") +
            "</b>. Edits create a hypothetical track whose true outcome is unknown.")
        state.update(last_result=None, last_error=None)
        output.clear_output(wait=False)
    selector.observe(load_example, names="value")
    button = widgets.Button(description="Predict hit label", button_style="success",
                            layout=widgets.Layout(width="190px"))
    def predict(_=None):
        output.clear_output(wait=True)
        state.update(last_result=None, last_error=None)
        with output:
            try:
                row = pd.DataFrame([{name: widget.value for name, widget in fields.items()}])
                result = score_demo(row, bundle)
                state["last_result"] = result
                label = "Hit" if int(result.iloc[0].predicted_label) else "Non-hit"
                print(f"Predicted label: {label}")
                print(f"Model hit score: {result.iloc[0].hit_score:.3f}")
                print(f"Decision threshold: {bundle['threshold']:.2f}")
                print("This is a historical classification score, not a calibrated future-hit probability.")
            except (ValueError, TypeError) as exc:
                state["last_error"] = str(exc)
                print(f"Input error: {exc}")
    button.on_click(predict)
    load_example()
    controls = widgets.VBox([selector, source_label] + list(fields.values()) + [button, output])
    return {"view": controls, "selector": selector, "fields": fields,
            "button": button, "state": state, "predict": predict}

def create_batch_controls(examples, bundle):
    """Optional Colab upload/download buttons; displaying them never opens a picker."""
    import ipywidgets as widgets
    from IPython.display import display
    from pathlib import Path

    state = {"result": score_demo(examples, bundle)}
    output = widgets.Output()
    upload = widgets.Button(description="Upload track CSV", layout=widgets.Layout(width="180px"))
    download = widgets.Button(description="Download predictions", layout=widgets.Layout(width="200px"))
    def upload_csv(_):
        with output:
            output.clear_output(wait=True)
            try:
                from google.colab import files
                incoming = files.upload()
                if not incoming:
                    print("No file selected.")
                    return
                if len(incoming) != 1:
                    raise ValueError("Choose one CSV file at a time.")
                frame = pd.read_csv(io.BytesIO(next(iter(incoming.values()))))
                if len(frame) > 10000:
                    raise ValueError("Please upload at most 10,000 tracks.")
                result = score_demo(frame, bundle)
                state["result"] = result
                print(f"Scored {len(result)} tracks.")
                display(result)
            except ImportError:
                print("File picker available in Google Colab. In local Jupyter, use score_demo(pd.read_csv(path), bundle).")
            except (ValueError, TypeError) as exc:
                print(f"CSV error: {exc}")
    def download_csv(_):
        path = Path("hitpredict_predictions.csv")
        state["result"].to_csv(path, index=False)
        try:
            from google.colab import files
            files.download(str(path))
        except ImportError:
            with output:
                print(f"Saved {path}")
    upload.on_click(upload_csv)
    download.on_click(download_csv)
    return {"view": widgets.VBox([widgets.HBox([upload, download]), output]),
            "state": state, "upload": upload, "download": download}


In [ ]:
# Embedded source-data snapshot; rebuilding requires scripts/build_colab_demo.py.
DEMO_ASSETS_B64 = (
    "..."
)
DEMO_ASSETS_SHA256 = '181adc94e12a03a0ec3e03c71a49ae184892627dd80e0cf3f14082620caeb6af'


In [ ]:
tracks, reference = decode_demo_assets(DEMO_ASSETS_B64, DEMO_ASSETS_SHA256)
summary = tracks.groupby('split').agg(tracks=('Label', 'size'), hits=('Label', 'sum'),
                                     groups=('group', 'nunique')).reindex(['train', 'validation', 'test'])
display(summary)
print('Loaded', len(tracks), 'tracks with verified group separation.')

## 3. Fit the selected model
This fits a random forest with 250 trees, maximum depth 16, and minimum leaf size 5 on the **2,554 training tracks only**. Imputation and scaling also fit only on training data. The original model comparison is documented in the analysis notebook. This demo does not repeat model selection or use test labels to tune its settings.

In [ ]:
from time import perf_counter
started = perf_counter()
bundle = fit_demo_model(tracks, reference)
print(f'Model ready in {perf_counter() - started:.1f} seconds.')
print(f'Decision threshold: {bundle["threshold"]:.2f}')

## 4. Evaluate on the held-out test tracks
“This runtime” reports newly computed metrics. “Original experiment” is the stored reference, not a claim that a different runtime must reproduce every value exactly. The confusion matrix rows are source labels and the columns are predictions.

In [ ]:
test_predictions, current_metrics = evaluate_demo(tracks, bundle)
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
comparison = pd.DataFrame({
    'This runtime': {k: current_metrics[k] for k in metric_names},
    'Original experiment': {k: reference['test'][k] for k in metric_names}
})
display(comparison.round(4))
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(np.array(current_metrics['confusion_matrix']),
                       display_labels=['Non-hit', 'Hit']).plot(ax=axes[0], cmap='Greens', colorbar=False)
axes[0].set_title('Current runtime: test confusion matrix')
RocCurveDisplay.from_predictions(test_predictions.Label, test_predictions.hit_score, ax=axes[1])
axes[1].set_title('Current runtime: test ROC curve')
plt.tight_layout()
plt.show()

## 5. Interactive track demo
Choose a held-out track, or choose **Custom track** and enter measured audio features. Click **Predict hit label**. Editing an example creates a hypothetical input; its original label does not establish the outcome of the edited song.

In [ ]:
examples = tracks.loc[tracks.split == 'test'].groupby('Label', group_keys=False).head(4).copy()
demo_controls = create_demo_controls(examples, bundle)
display(demo_controls['view'])

## 6. Batch demo and CSV download
The sample batch contains eight held-out tracks, including mistakes if the model makes them. **Download predictions** saves the current batch. **Upload track CSV** optionally scores another file with the nine feature columns listed below; names and labels are never model inputs.

In [ ]:
print('Required CSV columns:', ','.join(DEMO_FEATURES))
sample_batch = score_demo(examples, bundle)
display(sample_batch[['Artist', 'Track', 'Label', 'hit_score', 'predicted_label']].round(3))
batch_controls = create_batch_controls(examples, bundle)
display(batch_controls['view'])

## 7. Editable-code fallback
If the widget controls do not appear, this cell produces a prediction directly. Change the values and run this cell again.

In [ ]:
custom_track = pd.DataFrame([{
    'Danceability': 0.65, 'Energy': 0.70, 'Loudness': -6.0,
    'Speechiness': 0.05, 'Acousticness': 0.15, 'Instrumentalness': 0.001,
    'Liveness': 0.12, 'Valence': 0.60, 'Tempo': 120.0
}])
display(score_demo(custom_track, bundle).round(4))

## Sources and scope
Dataset: Georgieva, Suta and Burton, [HitPredict, Stanford CS229 (2018)](https://cs229.stanford.edu/proj2018/report/16.pdf), using the public CSV archive linked in the report. The cleaned snapshot removes 21 duplicate artist/title pairs. The source date-window descriptions are inconsistent, and labels are inherited rather than independently reconstructed.

The model uses nine audio features. Artist names define evaluation groups, and ArtistScore is excluded. Aliases, historical sampling, omitted marketing factors, and uncalibrated scores limit generalization to new releases. Instrumentalness importance in the original analysis is an association, not a causal conclusion.

[Colab notebook upload and runtime documentation](https://research.google.com/colaboratory/faq.html). The notebook runs the interaction directly within Colab without a separate web server.